# Dashboard Migration: Export & Transform (Manual Workflow)

## Overview

This notebook exports dashboards from source workspace and applies catalog/schema transformations.

### Manual Workflow

1. **This Notebook (Export & Transform)**: Export dashboards and apply transformations
2. **Manual Import**: Import dashboards to target workspace using your preferred method:
   - Option A: Bundle approach (recommended) - See Bundle/ folder
   - Option B: Databricks UI - Import .lvdash.json files manually
   - Option C: Custom SDK script
3. **Notebook 03A (Apply Permissions)**: Apply captured permissions to imported dashboards

**Note:** The automated SDK import (old Notebook 03) has been deprecated due to 5-minute timeout issues with large dashboards.

## What This Notebook Does

- 📤 Export dashboards from source workspace as .lvdash.json files
  - Option A: Load from inventory CSV (from Notebook 00)
  - Option B: Discover from folder path
  - Option C: Use explicit dashboard IDs
- 🔐 Capture dashboard permissions (ACLs)
- 🔄 Apply CSV mappings to transform catalog/schema/table references
- 📁 Save transformed dashboards to volume for manual import

## Prerequisites

✅ CSV mapping file created at `/Volumes/.../mappings/catalog_schema_mapping.csv`

---


In [ ]:
# Install/upgrade required packages
%pip install -U databricks-sdk pandas --quiet
dbutils.library.restartPython()

## Cell 1: Import Configuration from Notebook 1

**Purpose:** Re-import libraries and re-run configuration from Notebook 1.

**Instructions:** Copy the configuration values from Notebook 1 Cell 3 and Cell 4.

In [ ]:
# Import libraries
import json
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

# ============================================================================
# AUTHENTICATION CONFIGURATION (from Notebook 1)
# ============================================================================

AUTH_METHOD = "oauth"  # Options: "oauth", "service_principal", "pat"

SOURCE_WORKSPACE_URL = "https://your-source-workspace.cloud.databricks.com"
TARGET_WORKSPACE_URL = "https://your-target-workspace.cloud.databricks.com"

# Load credentials based on auth method
if AUTH_METHOD == "service_principal":
    SOURCE_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="source-sp-client-id")
    SOURCE_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="source-sp-secret")
    SOURCE_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="source-sp-tenant")
    TARGET_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="target-sp-client-id")
    TARGET_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="target-sp-secret")
    TARGET_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="target-sp-tenant")
elif AUTH_METHOD == "pat":
    SOURCE_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="source-token")
    TARGET_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="target-token")

# ============================================================================
# VOLUME CONFIGURATION (from Notebook 1)
# ============================================================================

VOLUME_BASE_PATH = "/Volumes/migration_catalog/migration_schema/migration_volume/dashboard_migration"
MAPPING_CSV_PATH = f"{VOLUME_BASE_PATH}/mappings/catalog_schema_mapping.csv"
EXPORT_PATH = f"{VOLUME_BASE_PATH}/exported"
TRANSFORMED_PATH = f"{VOLUME_BASE_PATH}/transformed"
LOGS_PATH = f"{VOLUME_BASE_PATH}/logs"

# Inventory CSV from Notebook 00 (optional)
INVENTORY_CSV_PATH = f"{VOLUME_BASE_PATH}/dashboard_inventory.csv"

# ============================================================================
# DASHBOARD SELECTION
# ============================================================================

# Choose one approach:

# Option 1: Explicit list of dashboard IDs
DASHBOARD_IDS = [
    # "dashboard_id_1",
    # "dashboard_id_2",
]

# Option 2: Export all dashboards from a folder
SOURCE_FOLDER_PATH = "/Workspace/Shared/Dashboards"

# Option 3: Use inventory CSV from Notebook 00
USE_INVENTORY_CSV = False  # Set to True to load from inventory CSV

# Select which option to use (only one should be True)
USE_FOLDER_PATH = False
USE_EXPLICIT_IDS = False

# ============================================================================
# OPTIONS
# ============================================================================

SKIP_PERMISSIONS = False  # Set True to skip permissions capture

print("✅ Configuration loaded")
print(f"   Auth method: {AUTH_METHOD}")
print(f"   Source: {SOURCE_WORKSPACE_URL}")
print(f"   Volume: {VOLUME_BASE_PATH}")
selection = 'Inventory CSV' if USE_INVENTORY_CSV else ('Folder' if USE_FOLDER_PATH else 'Explicit IDs')
print(f"   Selection: {selection}")

## Cell 2: Helper Functions - Authentication & Volume I/O

**Purpose:** Define helper functions for authentication and volume operations.

In [ ]:
def create_workspace_client(workspace_url: str, is_source: bool = True) -> WorkspaceClient:
    """Create WorkspaceClient with configured authentication."""
    if AUTH_METHOD == "service_principal":
        if is_source:
            return WorkspaceClient(
                host=workspace_url,
                client_id=SOURCE_SP_CLIENT_ID,
                client_secret=SOURCE_SP_CLIENT_SECRET,
                azure_tenant_id=SOURCE_SP_TENANT_ID
            )
        else:
            return WorkspaceClient(
                host=workspace_url,
                client_id=TARGET_SP_CLIENT_ID,
                client_secret=TARGET_SP_CLIENT_SECRET,
                azure_tenant_id=TARGET_SP_TENANT_ID
            )
    elif AUTH_METHOD == "oauth":
        # For OAuth, use notebook context (don't specify host for same workspace)
        return WorkspaceClient()
    elif AUTH_METHOD == "pat":
        token = SOURCE_PAT_TOKEN if is_source else TARGET_PAT_TOKEN
        return WorkspaceClient(host=workspace_url, token=token)
    else:
        raise ValueError(f"Unknown auth method: {AUTH_METHOD}")

def read_volume_file(volume_path: str) -> str:
    """Read file from volume (serverless-compatible)."""
    content = dbutils.fs.head(volume_path, 10485760)  # Read up to 10MB
    return content

def write_volume_file(volume_path: str, content: str) -> None:
    """Write file to volume (serverless-compatible)."""
    dbutils.fs.put(volume_path, content, overwrite=True)

def list_volume_files(volume_path: str, pattern: str = "*") -> List[str]:
    """List files in volume directory."""
    try:
        files = dbutils.fs.ls(volume_path)
        matched = []
        for file_info in files:
            file_path = file_info.path.replace("dbfs:", "")
            if pattern == "*" or file_path.endswith(pattern.replace("*", "")):
                matched.append(file_path)
        return matched
    except:
        return []

def read_csv_from_volume(csv_path: str) -> List[Dict[str, str]]:
    """Read CSV from volume."""
    content = read_volume_file(csv_path)
    lines = content.strip().split('\n')
    return list(csv.DictReader(lines))

def load_inventory_csv(csv_path: str) -> List[Dict]:
    """Load dashboard inventory from CSV generated by Notebook 00."""
    try:
        content = read_volume_file(csv_path)
        lines = content.strip().split('\n')
        reader = csv.DictReader(lines)
        
        dashboards = []
        for row in reader:
            dashboards.append({
                "id": row.get("dashboard_id", ""),
                "name": row.get("name", f"Dashboard_{row.get('dashboard_id', 'unknown')}"),
                "path": row.get("path", ""),
                "owner": row.get("owner", ""),
                "catalog_count": row.get("catalog_count", "0"),
                "table_count": row.get("table_count", "0"),
                "published": row.get("published", "False")
            })
        
        return dashboards
    except Exception as e:
        raise Exception(f"Failed to load inventory CSV: {e}")

print("✅ Helper functions loaded")

## Cell 3: Dashboard Export Helper Functions

**Purpose:** Functions for exporting dashboards and permissions from source workspace.

In [ ]:
def discover_dashboards_in_folder(client: WorkspaceClient, folder_path: str = None) -> List[Dict]:
    """Discover all Lakeview dashboards using Lakeview API."""
    dashboards = []
    try:
        # List all dashboards using Lakeview API
        dashboard_list = client.lakeview.list()
        for dashboard in dashboard_list:
            # Filter by parent_path if folder_path is provided
            if folder_path and hasattr(dashboard, 'parent_path'):
                if dashboard.parent_path != folder_path:
                    continue
            dashboards.append({
                "id": dashboard.dashboard_id,
                "path": getattr(dashboard, 'path', None),
                "name": dashboard.display_name or f"Dashboard_{dashboard.dashboard_id}"
            })
    except Exception as e:
        print(f"Warning: Could not list dashboards: {e}")
    return dashboards

def export_dashboard_to_lvdash(client: WorkspaceClient, dashboard_id: str) -> Dict:
    """Export dashboard as .lvdash.json using Lakeview API."""
    try:
        # Get dashboard using Lakeview SDK
        dashboard = client.lakeview.get(dashboard_id)
        serialized_dashboard = dashboard.serialized_dashboard
        
        if not serialized_dashboard:
            raise Exception(f"Dashboard {dashboard_id} has no serialized content")
        
        # Parse JSON to get display name
        dashboard_json = json.loads(serialized_dashboard)
        display_name = dashboard.display_name or dashboard_json.get('display_name', f'Dashboard_{dashboard_id}')
        
        return {
            "content": serialized_dashboard,
            "display_name": display_name,
            "dashboard_id": dashboard_id
        }
    except Exception as e:
        raise Exception(f"Failed to export dashboard {dashboard_id}: {e}")

def get_dashboard_permissions(client: WorkspaceClient, dashboard_id: str) -> Dict:
    """Get dashboard permissions using dashboard_id."""
    try:
        # Use "dashboards" for Lakeview (AI/BI) dashboards, not "dbsql-dashboards"
        perms = client.permissions.get("dashboards", dashboard_id)
        return {
            "access_control_list": [
                {
                    "user_name": acl.user_name,
                    "group_name": acl.group_name,
                    "service_principal_name": acl.service_principal_name,
                    "all_permissions": [
                        str(p.permission_level.value) if hasattr(p.permission_level, 'value') 
                        else str(p.permission_level) 
                        for p in (acl.all_permissions or [])
                    ]
                }
                for acl in (perms.access_control_list or [])
            ]
        }
    except Exception as e:
        print(f"   Warning: Could not fetch permissions: {e}")
        return {"access_control_list": []}

def save_dashboard_export(
    dashboard_id: str,
    lvdash_content: str,
    dashboard_name: str,
    permissions: Dict,
    volume_base_path: str
) -> Tuple[str, str]:
    """Save exported dashboard and permissions to volume."""
    # Clean name for filename
    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', dashboard_name)
    
    # Save dashboard
    dashboard_file = f"{volume_base_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
    write_volume_file(dashboard_file, lvdash_content)
    
    # Save permissions
    perms_file = f"{volume_base_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
    write_volume_file(perms_file, json.dumps(permissions, indent=2))
    
    return dashboard_file, perms_file

print("✅ Export helper functions loaded")

## Cell 4: Transform Helper Functions

**Purpose:** Functions for loading CSV mappings and transforming dashboard JSON.

In [ ]:
def load_mapping_csv(csv_path: str) -> List[Dict[str, str]]:
    """Load catalog/schema/table mappings from CSV."""
    return read_csv_from_volume(csv_path)

def find_and_replace_references(text: str, mappings: List[Dict]) -> str:
    """
    Replace catalog.schema.table references with proper boundary handling.
    
    Uses regex to handle:
    - Fully-qualified references: catalog.schema.table
    - Schema references: catalog.schema
    - Catalog-only references in JSON fields
    - Volume paths
    """
    if not isinstance(text, str):
        return text
    
    result = text
    
    for mapping in mappings:
        old_cat = mapping.get('old_catalog', '')
        old_schema = mapping.get('old_schema', '')
        old_table = mapping.get('old_table', '')
        new_cat = mapping.get('new_catalog', '')
        new_schema = mapping.get('new_schema', '')
        new_table = mapping.get('new_table', '')
        
        # Replace fully-qualified table references: catalog.schema.table
        if old_cat and old_schema and old_table:
            old_ref = f"{old_cat}.{old_schema}.{old_table}"
            new_ref = f"{new_cat}.{new_schema}.{new_table}"
            # Use word boundaries to avoid partial matches
            result = re.sub(rf'\b{re.escape(old_ref)}\b', new_ref, result)
        
        # Replace schema references: catalog.schema
        if old_cat and old_schema:
            old_ref = f"{old_cat}.{old_schema}"
            new_ref = f"{new_cat}.{new_schema}"
            # Negative lookahead: don't replace if followed by a dot (already handled above)
            result = re.sub(rf'\b{re.escape(old_ref)}(?!\.)', new_ref, result)
        
        # Replace catalog-only references (in JSON fields like "catalog": "old_catalog")
        if old_cat and new_cat:
            # Match catalog in quoted JSON fields: "catalog": "old_catalog"
            result = re.sub(
                rf'("catalog"\s*:\s*")({re.escape(old_cat)})(")',
                rf'\1{new_cat}\3',
                result
            )
            # Match catalog followed by dot (in qualified names)
            result = re.sub(rf'\b{re.escape(old_cat)}\.', f'{new_cat}.', result)
        
        # Replace volume paths
        old_vol = mapping.get('old_volume', '')
        new_vol = mapping.get('new_volume', '')
        if old_vol and new_vol:
            result = result.replace(f"/Volumes/{old_vol}/", f"/Volumes/{new_vol}/")
            result = result.replace(old_vol, new_vol)
    
    return result

def transform_dashboard_json(dashboard_json: str, mappings: List[Dict]) -> str:
    """Transform dashboard JSON using mappings."""
    # Parse JSON
    data = json.loads(dashboard_json)
    
    # Convert to string for replacement
    json_str = json.dumps(data, indent=2)
    
    # Apply mappings
    transformed_str = find_and_replace_references(json_str, mappings)
    
    # Validate JSON
    json.loads(transformed_str)  # Raises error if invalid
    
    return transformed_str

print("✅ Transform helper functions loaded")


## Cell 5: Export Dashboards from Source

**Purpose:** Export selected dashboards from source workspace.

**Process:**
1. Connect to source workspace
2. Discover or select dashboards
3. Export each dashboard
4. Capture permissions
5. Save to volume

In [ ]:
print("=" * 80)
print("📤 EXPORTING DASHBOARDS FROM SOURCE")
print("=" * 80)

# Create source client
source_client = create_workspace_client(SOURCE_WORKSPACE_URL, is_source=True)
print(f"\n✅ Connected to source workspace")

# Determine dashboards to export
dashboards_to_export = []

if USE_INVENTORY_CSV:
    print(f"\n📋 Loading dashboards from inventory CSV: {INVENTORY_CSV_PATH}")
    try:
        dashboards_to_export = load_inventory_csv(INVENTORY_CSV_PATH)
        print(f"   ✅ Loaded {len(dashboards_to_export)} dashboard(s) from inventory")
        
        # Display inventory summary
        print("\n   Inventory Summary:")
        df = pd.DataFrame(dashboards_to_export)
        display(df[['id', 'name', 'catalog_count', 'table_count', 'owner']])
        
    except Exception as e:
        print(f"   ❌ Failed to load inventory CSV: {e}")
        print(f"   💡 Make sure Notebook 00 has been run and CSV exists at: {INVENTORY_CSV_PATH}")
        raise
        
elif USE_FOLDER_PATH:
    print(f"\n🔍 Discovering dashboards in: {SOURCE_FOLDER_PATH}")
    dashboards_to_export = discover_dashboards_in_folder(source_client, SOURCE_FOLDER_PATH)
    print(f"   Found {len(dashboards_to_export)} dashboard(s)")
    
elif USE_EXPLICIT_IDS:
    print(f"\n📋 Using explicit dashboard IDs: {len(DASHBOARD_IDS)}")
    dashboards_to_export = [
        {"id": did, "path": None, "name": f"Dashboard_{did}"}
        for did in DASHBOARD_IDS
    ]
else:
    print("\n⚠️  No selection method enabled!")
    print("   Set USE_INVENTORY_CSV, USE_FOLDER_PATH, or USE_EXPLICIT_IDS to True")

if not dashboards_to_export:
    print("\n⚠️  No dashboards to export!")
    print("   Check your configuration or verify dashboards exist")
else:
    # Export each dashboard
    export_results = []
    
    for idx, dashboard_info in enumerate(dashboards_to_export, 1):
        dashboard_id = dashboard_info["id"]
        dashboard_path = dashboard_info.get("path") or f"/Workspace/dashboards/{dashboard_id}"
        
        print(f"\n[{idx}/{len(dashboards_to_export)}] Exporting: {dashboard_id}")
        
        try:
            # Export dashboard
            exported = export_dashboard_to_lvdash(source_client, dashboard_id)
            print(f"   ✅ Exported: {exported['display_name']}")
            
            # Get permissions
            if not SKIP_PERMISSIONS:
                permissions = get_dashboard_permissions(source_client, dashboard_id)
                acl_count = len(permissions.get('access_control_list', []))
                print(f"   🔐 Captured {acl_count} permission(s)")
            else:
                permissions = {"access_control_list": []}
                print(f"   ⊘ Permissions skipped")
            
            # Save to volume
            dashboard_file, perms_file = save_dashboard_export(
                dashboard_id=dashboard_id,
                lvdash_content=exported["content"],
                dashboard_name=exported["display_name"],
                permissions=permissions,
                volume_base_path=EXPORT_PATH
            )
            print(f"   💾 Saved to volume")
            
            export_results.append({
                "dashboard_id": dashboard_id,
                "dashboard_name": exported["display_name"],
                "dashboard_file": dashboard_file,
                "permissions_file": perms_file,
                "status": "success"
            })
            
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            export_results.append({
                "dashboard_id": dashboard_id,
                "status": "failed",
                "error": str(e)
            })
    
    # Summary
    successful = len([r for r in export_results if r["status"] == "success"])
    failed = len([r for r in export_results if r["status"] == "failed"])
    
    print("\n" + "=" * 80)
    print("📊 EXPORT SUMMARY")
    print("=" * 80)
    print(f"\n✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    print(f"\n📁 Exported files saved to: {EXPORT_PATH}")
    
    if successful > 0:
        print("\n👉 Ready for transformation! Continue to Cell 6.")

## Cell 6: Transform Dashboards

**Purpose:** Apply CSV mappings to transform exported dashboards.

**Process:**
1. Load CSV mappings
2. Read exported dashboards
3. Apply transformations
4. Save transformed dashboards

In [ ]:
print("=" * 80)
print("🔄 TRANSFORMING DASHBOARDS")
print("=" * 80)

# Load CSV mappings
print(f"\n📋 Loading CSV mappings from: {MAPPING_CSV_PATH}")
try:
    mappings = load_mapping_csv(MAPPING_CSV_PATH)
    print(f"   ✅ Loaded {len(mappings)} mapping rule(s)")
    
    # Display sample mappings
    print("\n   Sample mappings:")
    for i, mapping in enumerate(mappings[:3], 1):
        old_ref = f"{mapping['old_catalog']}.{mapping['old_schema']}.{mapping['old_table']}"
        new_ref = f"{mapping['new_catalog']}.{mapping['new_schema']}.{mapping['new_table']}"
        print(f"      {i}. {old_ref} → {new_ref}")
    if len(mappings) > 3:
        print(f"      ... and {len(mappings) - 3} more")
        
except Exception as e:
    print(f"   ❌ Failed to load CSV: {e}")
    raise

# Get exported dashboard files
print(f"\n🔍 Finding exported dashboards in: {EXPORT_PATH}")
exported_files = list_volume_files(EXPORT_PATH, "*.lvdash.json")
print(f"   Found {len(exported_files)} file(s)")

if not exported_files:
    print("\n⚠️  No exported dashboards found!")
    print("   Make sure Cell 5 (Export) completed successfully")
else:
    # Transform each dashboard
    transform_results = []
    
    for idx, dashboard_file in enumerate(exported_files, 1):
        filename = Path(dashboard_file).name
        print(f"\n[{idx}/{len(exported_files)}] Transforming: {filename}")
        
        try:
            # Read dashboard JSON
            dashboard_json = read_volume_file(dashboard_file)
            original_size = len(dashboard_json)
            
            # Apply transformations
            transformed_json = transform_dashboard_json(dashboard_json, mappings)
            transformed_size = len(transformed_json)
            
            # Save transformed dashboard
            transformed_file = f"{TRANSFORMED_PATH}/{filename}"
            write_volume_file(transformed_file, transformed_json)
            
            changes = "modified" if original_size != transformed_size else "no changes"
            print(f"   ✅ Transformed ({changes})")
            print(f"   💾 Saved to: {transformed_file}")
            
            transform_results.append({
                "filename": filename,
                "original_file": dashboard_file,
                "transformed_file": transformed_file,
                "status": "success",
                "changes": changes
            })
            
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            transform_results.append({
                "filename": filename,
                "status": "failed",
                "error": str(e)
            })
    
    # Summary
    successful = len([r for r in transform_results if r["status"] == "success"])
    failed = len([r for r in transform_results if r["status"] == "failed"])
    
    print("\n" + "=" * 80)
    print("📊 TRANSFORMATION SUMMARY")
    print("=" * 80)
    print(f"\n✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    print(f"\n📁 Transformed files saved to: {TRANSFORMED_PATH}")
    
    if successful > 0:
        print("\n" + "=" * 80)
        print("✅ EXPORT & TRANSFORM COMPLETE!")
        print("=" * 80)
        print("\n👉 Next Step: Open Notebook 3: 03_Import_and_Migrate.ipynb")
        print("\n   Notebook 3 will:")
        print("   - Import transformed dashboards to target workspace")
        print("   - Restore permissions")
        print("   - Generate migration report")

## Export & Transform Complete! 🎉

### What You've Accomplished

- ✅ Exported dashboards from source workspace
- ✅ Captured dashboard permissions
- ✅ Applied CSV mappings to transform references
- ✅ Saved transformed dashboards to volume

### Files Created

- **Exported**: `{EXPORT_PATH}/`
  - Dashboard .lvdash.json files
  - Permissions .json files

- **Transformed**: `{TRANSFORMED_PATH}/`
  - Transformed dashboard .lvdash.json files (ready for import)

### Next Steps

1. **Review transformed dashboards** (optional)
2. **Open Notebook 3**: `03_Import_and_Migrate.ipynb`
3. Choose your import workflow:
   - **Manual**: Import via Databricks UI, then apply ACLs
   - **Automated**: Programmatic import and ACL application

---

**Questions or Issues?** Refer to the comprehensive guide document.